In [1]:
import os
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [2]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(20),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.8, 1.2)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [4]:
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

for parameter in model.features.parameters():
    parameter.requires_grad = False

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\SAKTHI/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:04<00:00, 3.50MB/s]


In [5]:
input_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Linear(input_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, len(train_dataset.classes))
)

model = model.to(device)

In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=1e-4
)

print(model)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [7]:
EPOCHS = 15

history = {
    "accuracy": [],
    "val_accuracy": [],
    "loss": [],
    "val_loss": []
}

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total
    val_loss /= val_total
    val_accuracy = val_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch [1/15] Loss: 3.2338 Accuracy: 0.0654 Val Loss: 3.2002 Val Accuracy: 0.1564
Epoch [2/15] Loss: 3.1315 Accuracy: 0.2066 Val Loss: 3.1077 Val Accuracy: 0.3256
Epoch [3/15] Loss: 2.9943 Accuracy: 0.3176 Val Loss: 2.9929 Val Accuracy: 0.3231
Epoch [4/15] Loss: 2.8534 Accuracy: 0.3577 Val Loss: 2.8636 Val Accuracy: 0.3462
Epoch [5/15] Loss: 2.6912 Accuracy: 0.4071 Val Loss: 2.7425 Val Accuracy: 0.3667
Epoch [6/15] Loss: 2.5510 Accuracy: 0.4088 Val Loss: 2.6383 Val Accuracy: 0.3590
Epoch [7/15] Loss: 2.4344 Accuracy: 0.4346 Val Loss: 2.5360 Val Accuracy: 0.3718
Epoch [8/15] Loss: 2.3155 Accuracy: 0.4527 Val Loss: 2.4545 Val Accuracy: 0.3615
Epoch [9/15] Loss: 2.2024 Accuracy: 0.4681 Val Loss: 2.3973 Val Accuracy: 0.3718
Epoch [10/15] Loss: 2.1419 Accuracy: 0.4703 Val Loss: 2.3370 Val Accuracy: 0.3769
Epoch [11/15] Loss: 2.0477 Accuracy: 0.5132 Val Loss: 2.2869 Val Accuracy: 0.3538
Epoch [12/15] Loss: 1.9818 Accuracy: 0.5088 Val Loss: 2.2385 Val Accuracy: 0.4231
Epoch [13/15] Loss: 1.947

In [8]:
torch.save(
    model.state_dict(),
    "models/mobilenetv2_final.pth"
)

print("\n✅ MobileNetV2 model saved successfully")


✅ MobileNetV2 model saved successfully
